# 3 · Create the Contract Search Tool

The agent in [`agent/`](./agent) is built with the **Agent Bricks CLI**. Its tools are
declared in `agent/agent.toml`:

1. **Genie agent**: the Genie space from `2_genie/0` answers sales and campaign analytics questions.
2. **`search_contracts`**: a **Unity Catalog function** that wraps the Vector Search index
   from notebook `1` with the `vector_search()` SQL function. This notebook creates it.

A UC function keeps the retrieval logic governed in Unity Catalog: the agent can only call it
if its service principal has `EXECUTE` on the function and `SELECT` on the index
(granted in notebook `4`).

**What this notebook does**
1. Creates (or replaces) `search_contracts(query)` in the demo schema
2. Tests it with a sample question
3. Resolves the Genie space id and prints the exact `agentbricks` commands for the next step

### Define Parameters

| Widget | Default | What it controls |
| --- | --- | --- |
| `CATALOG_NAME` | `workspace` | Unity Catalog for demo objects. |
| `SCHEMA_NAME` | `marketing_campaign` | Schema with the Vector Search index. |
| `INDEX_NAME` | `contract_chunks_index` | Vector Search index from notebook `1`. |
| `FUNCTION_NAME` | `search_contracts` | Name of the UC function tool. |
| `NUM_RESULTS` | `5` | Chunks returned per search. |
| `GENIE_SPACE_TITLE` | `Marketing Campaign Analytics — Genie Space` | Title from `2_genie/0`, used to look up the space id. |

In [ ]:
dbutils.widgets.text("CATALOG_NAME", "workspace", "CATALOG_NAME — Unity Catalog catalog for demo objects")
dbutils.widgets.text("SCHEMA_NAME", "marketing_campaign", "SCHEMA_NAME — schema inside the catalog")
dbutils.widgets.text("INDEX_NAME", "contract_chunks_index", "INDEX_NAME — Vector Search index (from notebook 1)")
dbutils.widgets.text("FUNCTION_NAME", "search_contracts", "FUNCTION_NAME — UC function tool name")
dbutils.widgets.text("NUM_RESULTS", "5", "NUM_RESULTS — chunks returned per search")
dbutils.widgets.text("GENIE_SPACE_TITLE", "Marketing Campaign Analytics — Genie Space", "GENIE_SPACE_TITLE — title used in 2_genie/0")

CATALOG_NAME = dbutils.widgets.get("CATALOG_NAME")
SCHEMA_NAME = dbutils.widgets.get("SCHEMA_NAME")
INDEX_NAME = dbutils.widgets.get("INDEX_NAME")
FUNCTION_NAME = dbutils.widgets.get("FUNCTION_NAME")
NUM_RESULTS = int(dbutils.widgets.get("NUM_RESULTS"))
GENIE_SPACE_TITLE = dbutils.widgets.get("GENIE_SPACE_TITLE").strip()

INDEX_FQN = f"{CATALOG_NAME}.{SCHEMA_NAME}.{INDEX_NAME}"
FUNCTION_FQN = f"{CATALOG_NAME}.{SCHEMA_NAME}.{FUNCTION_NAME}"
print("Index    :", INDEX_FQN)
print("Function :", FUNCTION_FQN)

### Create the UC Function

The function comment and parameter comment are what the model sees as the tool
description, so they explain when to call it.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {FUNCTION_FQN}(
  query STRING COMMENT 'Natural-language question or keywords about the contracts, for example: payment terms of the Echo agreement'
)
RETURNS TABLE (source_uri STRING, chunk_to_retrieve STRING, score DOUBLE)
COMMENT 'Searches the marketing services contracts (parties, scope, fees, payment terms, dates, termination, obligations). Returns the most relevant passages with the contract file name in source_uri.'
RETURN
  SELECT source_uri, chunk_to_retrieve, search_score AS score
  FROM vector_search(
    index => '{INDEX_FQN}',
    query_text => query,
    num_results => {NUM_RESULTS}
  )
""")
print("Created", FUNCTION_FQN)

### Test the Function

In [ ]:
display(spark.sql(f"SELECT * FROM {FUNCTION_FQN}('What are the payment terms in the Echo contract?')"))

### Next Step: Bind the Tools and Deploy the Agent

The Agent Bricks CLI runs from a terminal (your laptop, or the **web terminal** of a
Databricks cluster). The next cell resolves the Genie space id and prints the commands with
your values filled in. Both tools use `--auth app`, so they run as the agent's service
principal; notebook `4` grants it access.

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
# REST call so this works with any installed databricks-sdk version.
spaces, page_token = [], None
while True:
    query = {"page_size": 100, **({"page_token": page_token} if page_token else {})}
    resp = w.api_client.do("GET", "/api/2.0/genie/spaces", query=query)
    spaces.extend(resp.get("spaces", []))
    page_token = resp.get("next_page_token")
    if not page_token:
        break
match = next((s for s in spaces if s.get("title") == GENIE_SPACE_TITLE), None)
assert match, f"No Genie space titled '{GENIE_SPACE_TITLE}'. Run 2_genie/0_CreateGenieSpace first."
GENIE_SPACE_ID = match["space_id"]

print(f"""Run these from the 3_agent-bricks/agent folder:

  databricks auth login --host {w.config.host} --profile <profile>
  agentbricks login --profile <profile>
  agentbricks tools add genie-agent {GENIE_SPACE_ID} --name sales_analytics --auth app
  agentbricks tools add uc-function {FUNCTION_FQN} --name search_contracts --auth app
  agentbricks deploy

Then run notebook 4_GrantAgentAccess.""")